In [5]:
import sys

sys.path.append(r"E:\Projects\EduAI\backend")

In [6]:

from __future__ import annotations

import json
from datetime import datetime, timezone
from typing import Annotated, TypedDict, List, Optional

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, SystemMessage
from app.agents.llm import llm


In [7]:
_COLUMNS = [
    "id", "classroom_id", "event_date", "event_type", "reference_id",
    "created_by", "is_personal", "title", "description",
    "created_at", "updated_at",
]


In [8]:
EVENT_TYPES = ["TASK"]


In [9]:
def _row_to_dict(row) -> dict:
    d = dict(zip(_COLUMNS, row))
    for key in ("event_date", "created_at", "updated_at"):
        if isinstance(d.get(key), datetime):
            d[key] = d[key].isoformat()
    return d

In [10]:
def fetch_all_my_events(conn, user_id: int, classroom_id: int) -> dict:
    """Fetch all classroom-wide events for `classroom_id`, plus the current
    user's own personal events within that classroom."""
    curr = conn.cursor()
    try:
        curr.execute(
            f"""
            SELECT {", ".join(_COLUMNS)}
            FROM calendar_events
            WHERE classroom_id = %s AND is_personal = FALSE
            ORDER BY event_date;
            """,
            (classroom_id,),
        )
        classroom_rows = curr.fetchall()

        curr.execute(
            f"""
            SELECT {", ".join(_COLUMNS)}
            FROM calendar_events
            WHERE classroom_id = %s AND is_personal = TRUE AND created_by = %s
            ORDER BY event_date;
            """,
            (classroom_id, user_id),
        )
        personal_rows = curr.fetchall()
        print(f"fetch_all_my_events: classroom_rows={len(classroom_rows)}, personal_rows={len(personal_rows)}")

        return {
            "success": True,
            "message": "Events fetched successfully",
            "data": {
                "classroom_events": [_row_to_dict(r) for r in classroom_rows],
                "personal_events": [_row_to_dict(r) for r in personal_rows],
            },
        }

    except Exception as e:
        conn.rollback()
        print(f"fetch_all_my_events failed, rolled back. Error: {e}")
        return {"success": False, "message": f"Failed to fetch events: {e}", "data": None}

    finally:
        curr.close()
